In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv
/kaggle/input/competitions/comment-category-prediction-challenge/train.csv
/kaggle/input/competitions/comment-category-prediction-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
train = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/test.csv")

df = train.copy()
y  = df['label']
df = df.drop(columns=['label'])
test_df = test.copy()

#1) Date Feature Engineering
for d in [df, test_df]:
    d['created_date'] = pd.to_datetime(d['created_date'])
    d['year']  = d['created_date'].dt.year
    d['month'] = d['created_date'].dt.month
    d['hour']  = d['created_date'].dt.hour
    d.drop(columns=['created_date'], inplace=True)

#2) Handle Missing Values
for d in [df, test_df]:
    d['race'].fillna('Unknown', inplace=True)
    d['religion'].fillna('Unknown', inplace=True)
    d['gender'].fillna('Unknown', inplace=True)
    d['comment'].fillna('', inplace=True)
    d['disability'] = d['disability'].astype(int)

#3,4) Preprocessing Pipeline
preprocessor = ColumnTransformer(transformers=[
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000), 'comment'),
    ('ohe',   OneHotEncoder(drop='first', handle_unknown='ignore'), ['race', 'religion', 'gender'])
], remainder='passthrough')

X      = preprocessor.fit_transform(df)
X_test = preprocessor.transform(test_df)

#5) Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

#answers
print(f"Q1) Max TF-IDF features         : 5000")
print(f"Q2) Features in X_train         : {X_train.shape[1]}")
print(f"Q3) Features in X_val           : {X_val.shape[1]}")

# Q4 & Q5
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred_nb = mnb.predict(X_val)
print(f"Q4) MNB Validation Accuracy     : {accuracy_score(y_val, y_pred_nb):.3f}")
report_nb = classification_report(y_val, y_pred_nb, output_dict=True)
print(f"Q5) Precision label 3 (MNB)     : {report_nb['3']['precision']:.2f}")

# Q6, Q7, Q8
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
print(f"Q6) LR Validation Accuracy      : {accuracy_score(y_val, lr.predict(X_val)):.3f}")
print(f"Q7) LR Training Accuracy        : {accuracy_score(y_train, lr.predict(X_train)):.3f}")
report_lr = classification_report(y_val, lr.predict(X_val), output_dict=True)
print(f"Q8) Precision label 1 (LR)      : {report_lr['1']['precision']:.2f}")

# Q9 & Q10
gs = GridSearchCV(
    LogisticRegression(solver='liblinear', max_iter=500, random_state=42),
    {'C': [0.1, 1, 10]}, cv=3, n_jobs=-1)
gs.fit(X_train, y_train)
print(f"Q9)  Best C                      : {gs.best_params_['C']}")
print(f"Q10) Tuned Validation Accuracy   : {accuracy_score(y_val, gs.best_estimator_.predict(X_val)):.2f}")


/tmp/ipykernel_55/713411667.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  d['race'].fillna('Unknown', inplace=True)
/tmp/ipykernel_55/713411667.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.m

Q1) Max TF-IDF features         : 5000
Q2) Features in X_train         : 5031
Q3) Features in X_val           : 5031
Q4) MNB Validation Accuracy     : 0.808
Q5) Precision label 3 (MNB)     : 0.52
Q6) LR Validation Accuracy      : 0.839
Q7) LR Training Accuracy        : 0.841
Q8) Precision label 1 (LR)      : 0.62


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Q9)  Best C                      : 10
Q10) Tuned Validation Accuracy   : 0.87
